In [1]:
from __future__ import annotations

import os
# os.environ["HWLOC_HIDE_ERRORS"] = "1"
# os.environ["TF_ENABLE_NUMA"] = "0"
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"
# 3. Provide valid temporary directory for ptxas CUDA kernel compilation
os.environ["TMPDIR"] = "/tmp"
os.environ["TMP"] = "/tmp"
os.environ["TEMP"] = "/tmp"

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
print("Jax version", jax.__version__, "64-bit", jax.config.read("jax_enable_x64"))
print("jax", jax.__version__, "GPU backend:", jax.default_backend() == 'gpu', f"({jax.devices('gpu')[0].device_kind})" if jax.default_backend() == 'gpu' else "")
import jax.numpy as jnp
import s2fft
from tqdm.auto import tqdm
from typing import Literal

import jax.numpy as jnp
import s2fft
import argparse
import shutil
from pathlib import Path

def resample_mw_to_mwss(map_path: Path, output_path: Path) -> None:
    m = np.load(map_path)
    m = m.reshape((1500, 2999, 5)).astype(np.float32)

    m_mwss = np.stack(
        [
            np.array(s2fft.utils.resampling_jax.mw_to_mwss(m[..., c], L=1500)).real
            for c in range(5)
        ],
        axis=-1,
    )

    np.save(output_path, m_mwss.astype(np.float64, copy=False))

def L_downsample_mwss(map_path: Path, output_path: Path, L_in: int, L_out: int) -> None:
    m = np.load(map_path)
    # m_halfL = np.stack(
    #         [
    #             np.array(s2fft.utils.resampling.downsample_by_two_mwss(m[..., c], L=L_in)).real
    #             for c in range(5)
    #         ], 
    #     axis=-1
    #     )  # shape: (L_in/2, ...)
    # m_halfL = np.array(s2fft.utils.resampling.downsample_by_two_mwss(m, L=L_in)).real
    # m = m[0:-1:2, 0:-1:2, :]
    def L_downsample(f_map, L_in, L_out):
        L_cut = L_in-L_out
        m = s2fft.transforms.spherical.forward(f_map, L=L_in, reality=True, method='jax_cuda', sampling='mwss')
        m = m[:L_out, L_cut:-L_cut]  # keep modes l < 750
        m = s2fft.transforms.spherical.inverse(m, L=L_out, reality=True, method='jax_cuda', sampling='mwss')
        return m
    m = np.stack(
            [
                np.array(L_downsample(m[..., c], L_in=L_in, L_out=L_out)).real
                for c in range(5)
            ], 
        axis=-1
        )  # shape: (L_in/2, ...)

    np.save(output_path, m.astype(np.float64, copy=False))
    del m
    # raise NotImplementedError("L_downsample_by_two_mwss is not implemented yet.")



Jax version 0.11.0 64-bit True


E0811 15:24:34.999037   38380 numa_hwloc.cc:121] Call to hwloc_set_cpubind() failed: Invalid argument [22]


jax 0.11.0 GPU backend: True (NVIDIA GH200 120GB)


Downsample

In [3]:
SOURCE_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "5_tomo_bins_flm_1500_batch_3_mwss/"
)
OUTPUT_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "5_tomo_bins_flm_750_batch_3_mwss/"
)
OVERWRITE = False


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    params_src = SOURCE_DIR / "params.npy"
    params_dst = OUTPUT_DIR / "params.npy"
    if params_src.exists() and (OVERWRITE or not params_dst.exists()):
        shutil.copy2(params_src, params_dst)

    map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
    if not map_files:
        raise FileNotFoundError(f"No map_*.npy files found in {SOURCE_DIR}")
    
    converted = 0
    skipped = 0

    for map_path in tqdm(map_files, desc="Resampling maps"):
        output_path = OUTPUT_DIR / map_path.name
        if output_path.exists() and not OVERWRITE:
            skipped += 1
            continue

        # resample_mw_to_mwss(map_path, output_path)
        L_downsample_mwss(map_path, output_path, L_in=1500, L_out=750)
        converted += 1

    print(
        f"Converted {converted} maps, skipped {skipped} existing maps.\n"
        f"Source: {SOURCE_DIR}\n"
        f"Output: {OUTPUT_DIR}"
    )

print("SOURCE_DIR =", SOURCE_DIR)
print("Exists     =", SOURCE_DIR.exists())
print("Is dir     =", SOURCE_DIR.is_dir())
map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
print("Count      =", len(map_files))
print("First 5    =", [p.name for p in map_files[:5]])

main()

SOURCE_DIR = /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_3_mwss
Exists     = True
Is dir     = True
Count      = 2000
First 5    = ['map_0.npy', 'map_1.npy', 'map_10.npy', 'map_100.npy', 'map_1000.npy']


Resampling maps:   0%|                                                                                                  | 0/2000 [00:00<?, ?it/s]

Resampling maps: 100%|█████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [2:23:19<00:00,  4.30s/it]

Converted 2000 maps, skipped 0 existing maps.
Source: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_3_mwss
Output: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_batch_3_mwss


Compute data stats


In [3]:
import os
import glob
import numpy as np
from tqdm import tqdm

def compute_and_save_map_stats(directory, output_filename="map_stats.npz"):
    """
    Computes channel-wise mean and std across all map_{i}.npy files in `directory`
    loading only one map into memory at a time. Saves stats into `output_filename`.
    """
    map_files = glob.glob(os.path.join(directory, "map_*.npy"))
    n_files = len(map_files)
    assert n_files > 0, f"No map_*.npy files found in {directory}"

    print(f"Found {n_files} map files. Calculating mean and std...")

    # Inspect shape of first map to initialize accumulators
    sample_map = np.load(map_files[0])
    num_channels = sample_map.shape[-1]
    
    # Online accumulators for each channel
    count = 0
    mean = np.zeros(num_channels, dtype=np.float64)
    M2 = np.zeros(num_channels, dtype=np.float64)

    # Welford's algorithm over each map loading only 1 map at a time
    for map_path in tqdm(map_files):
        m = np.load(map_path).astype(np.float64)
        # Collapse spatial dimensions (theta, phi) -> reshape to (-1, channels)
        pixels = m.reshape(-1, num_channels)
        
        n_pixels = pixels.shape[0]
        batch_mean = np.mean(pixels, axis=0)
        batch_var = np.var(pixels, axis=0)

        # Update global accumulators (Welford / Parallel algorithm)
        if count == 0:
            mean = batch_mean
            M2 = batch_var * n_pixels
            count = n_pixels
        else:
            delta = batch_mean - mean
            new_count = count + n_pixels
            mean += delta * n_pixels / new_count
            M2 += batch_var * n_pixels + (delta ** 2) * count * n_pixels / new_count
            count = new_count

    variance = M2 / count
    std = np.sqrt(variance)

    # Cast stats back to float32 for model training
    mean = mean.astype(np.float32)
    std = std.astype(np.float32)

    stats_path = os.path.join(directory, output_filename)
    np.savez(stats_path, mean=mean, std=std)
    print(f"Successfully saved stats to {stats_path}")
    print("Channel Mean:", mean)
    print("Channel Std :", std)
    return stats_path

# Execute calculation for your map directory
data_dir = "/projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_all_mwss/"
compute_and_save_map_stats(data_dir)


Found 6000 map files. Calculating mean and std...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 6000/6000 [07:19<00:00, 13.64it/s]

Successfully saved stats to /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_all_mwss/map_stats.npz
Channel Mean: [0.00050942 0.00050919 0.00050897 0.00050895 0.00050911]
Channel Std : [0.0081269  0.00812699 0.00812699 0.00812639 0.00812658]


'/projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_all_mwss/map_stats.npz'

MW to MWSS resampling

In [2]:
SOURCE_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "map_compression_samples_5_tomo_bins_flm_1500_batch_3/"
)
OUTPUT_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "5_tomo_bins_flm_1500_batch_3_mwss/"
)
SKIP = []


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Source directory : {SOURCE_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")

    params_src = SOURCE_DIR / "params.npy"
    params_dst = OUTPUT_DIR / "params.npy"
    if params_src.exists():
        if params_dst.exists():
            print(f"params.npy already exists at {params_dst}; skipping copy.")
        else:
            print(f"Copying params.npy -> {params_dst}")
            shutil.copy2(params_src, params_dst)
    else:
        print(f"Warning: {params_src} not found.")

    map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
    if not map_files:
        raise FileNotFoundError(f"No map_*.npy files found in {SOURCE_DIR}")

    print(f"Found {len(map_files)} map files to process.")

    skipped = 0
    converted = 0

    for i, map_path in enumerate(
        tqdm(map_files, desc="Resampling maps", unit="map", leave=True),
        start=1,
    ):
        output_path = OUTPUT_DIR / map_path.name
        if output_path.exists():
            skipped += 1
            print(f"[{i}/{len(map_files)}] Skipping existing file: {map_path.name}")
            continue
        elif map_path.name in SKIP:
            skipped += 1
            print(f"[{i}/{len(map_files)}] Skipping file in SKIP list: {map_path.name}")
            continue

        print(f"[{i}/{len(map_files)}] Converting {map_path.name} -> {output_path.name}")
        resample_mw_to_mwss(map_path, output_path)
        converted += 1
        # print(f"  Done. Converted so far: {converted}, skipped so far: {skipped}")

    print(
        f"Finished. Converted {converted} maps, skipped {skipped} existing maps.\n"
        f"Source: {SOURCE_DIR}\n"
        f"Output: {OUTPUT_DIR}"
    )

main()

Source directory : /projects/u6pf/brianycc/spherical_maps/map_compression_samples_5_tomo_bins_flm_1500_batch_3
Output directory: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_3_mwss
params.npy already exists at /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_3_mwss/params.npy; skipping copy.
Found 2000 map files to process.


Resampling maps:   0%|                                                                                                 | 0/2000 [00:00<?, ?map/s]

Resampling maps:  17%|██████████████▍                                                                      | 339/2000 [00:00<00:00, 3381.16map/s]

[1/2000] Skipping existing file: map_0.npy
[2/2000] Skipping existing file: map_1.npy
[3/2000] Skipping existing file: map_10.npy
[4/2000] Skipping existing file: map_100.npy
[5/2000] Skipping existing file: map_1000.npy
[6/2000] Skipping existing file: map_1001.npy
[7/2000] Skipping existing file: map_1002.npy
[8/2000] Skipping existing file: map_1003.npy
[9/2000] Skipping existing file: map_1004.npy
[10/2000] Skipping existing file: map_1005.npy
[11/2000] Skipping existing file: map_1006.npy
[12/2000] Skipping existing file: map_1007.npy
[13/2000] Skipping existing file: map_1008.npy
[14/2000] Skipping existing file: map_1009.npy
[15/2000] Skipping existing file: map_101.npy
[16/2000] Skipping existing file: map_1010.npy
[17/2000] Skipping existing file: map_1011.npy
[18/2000] Skipping existing file: map_1012.npy
[19/2000] Skipping existing file: map_1013.npy
[20/2000] Skipping existing file: map_1014.npy
[21/2000] Skipping existing file: map_1015.npy
[22/2000] Skipping existing file

Resampling maps:  35%|█████████████████████████████▉                                                       | 705/2000 [00:00<00:00, 3543.21map/s]

...

[706/2000] Skipping existing file: map_1632.npy
[707/2000] Skipping existing file: map_1633.npy
[708/2000] Skipping existing file: map_1634.npy
[709/2000] Skipping existing file: map_1635.npy
[710/2000] Skipping existing file: map_1636.npy
[711/2000] Skipping existing file: map_1637.npy
[712/2000] Skipping existing file: map_1638.npy
[713/2000] Skipping existing file: map_1639.npy
[714/2000] Skipping existing file: map_164.npy
[715/2000] Skipping existing file: map_1640.npy
[716/2000] Skipping existing file: map_1641.npy
[717/2000] Skipping existing file: map_1642.npy
[718/2000] Skipping existing file: map_1643.npy
[719/2000] Skipping existing file: map_1644.npy
[720/2000] Skipping existing file: map_1645.npy
[721/2000] Skipping existing file: map_1646.npy
[722/2000] Skipping existing file: map_1647.npy
[723/2000] Skipping existing file: map_1648.npy
[724/2000] Skipping existing file: map_1649.npy
[725/2000] Skipping existing file: map_165.npy
[726/2000] Skipping existing file: map_165

Resampling maps:  72%|████████████████████████████████████████████████████████████▌                       | 1441/2000 [00:00<00:00, 3631.70map/s]

[1391/2000] Skipping existing file: map_45.npy
[1392/2000] Skipping existing file: map_450.npy
[1393/2000] Skipping existing file: map_451.npy
[1394/2000] Skipping existing file: map_452.npy
[1395/2000] Skipping existing file: map_453.npy
[1396/2000] Skipping existing file: map_454.npy
[1397/2000] Skipping existing file: map_455.npy
[1398/2000] Skipping existing file: map_456.npy
[1399/2000] Skipping existing file: map_457.npy
[1400/2000] Skipping existing file: map_458.npy
[1401/2000] Skipping existing file: map_459.npy
[1402/2000] Skipping existing file: map_46.npy
[1403/2000] Skipping existing file: map_460.npy
[1404/2000] Skipping existing file: map_461.npy
[1405/2000] Skipping existing file: map_462.npy
[1406/2000] Skipping existing file: map_463.npy
[1407/2000] Skipping existing file: map_464.npy
[1408/2000] Skipping existing file: map_465.npy
[1409/2000] Skipping existing file: map_466.npy
[1410/2000] Skipping existing file: map_467.npy
[1411/2000] Skipping existing file: map_46

Resampling maps: 100%|████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 3615.54map/s]

[1446/2000] Skipping existing file: map_5.npy
[1447/2000] Skipping existing file: map_50.npy
[1448/2000] Skipping existing file: map_500.npy
[1449/2000] Skipping existing file: map_501.npy
[1450/2000] Skipping existing file: map_502.npy
[1451/2000] Skipping existing file: map_503.npy
[1452/2000] Skipping existing file: map_504.npy
[1453/2000] Skipping existing file: map_505.npy
[1454/2000] Skipping existing file: map_506.npy
[1455/2000] Skipping existing file: map_507.npy
[1456/2000] Skipping existing file: map_508.npy
[1457/2000] Skipping existing file: map_509.npy
[1458/2000] Skipping existing file: map_51.npy
[1459/2000] Skipping existing file: map_510.npy
[1460/2000] Skipping existing file: map_511.npy
[1461/2000] Skipping existing file: map_512.npy
[1462/2000] Skipping existing file: map_513.npy
[1463/2000] Skipping existing file: map_514.npy
[1464/2000] Skipping existing file: map_515.npy
[1465/2000] Skipping existing file: map_516.npy
[1466/2000] Skipping existing file: map_517.

In [ ]:
# def main() -> None:
#     parser = argparse.ArgumentParser(
#         description="Bulk-resample spherical maps from MW to MWSS."
#     )
#     parser.add_argument(
#         "--source-dir",
#         type=Path,
#         default=Path(
#             "/projects/u6pf/brianycc/spherical_maps/"
#             "map_compression_samples_5_tomo_bins_flm_1500_batch_1"
#         ),
#         help="Directory containing map_*.npy and params.npy",
#     )
#     parser.add_argument(
#         "--output-dir",
#         type=Path,
#         default=Path(
#             "/projects/u6pf/brianycc/spherical_maps/"
#             "map_compression_samples_5_tomo_bins_flm_1500_batch_1_mwss"
#         ),
#         help="Directory where MWSS-resampled files will be written",
#     )
#     args = parser.parse_args()

#     source_dir: Path = args.source_dir
#     output_dir: Path = args.output_dir
#     output_dir.mkdir(parents=True, exist_ok=True)

#     params_src = source_dir / "params.npy"
#     params_dst = output_dir / "params.npy"
#     if params_src.exists() and not params_dst.exists():
#         shutil.copy2(params_src, params_dst)

#     map_files = sorted(source_dir.glob("map_*.npy"))
#     if not map_files:
#         raise FileNotFoundError(f"No map_*.npy files found in {source_dir}")

#     skipped = 0
#     converted = 0

#     for map_path in map_files:
#         output_path = output_dir / map_path.name
#         if output_path.exists():
#             skipped += 1
#             continue
#         resample_mw_to_mwss(map_path, output_path)
#         converted += 1

#     print(
#         f"Converted {converted} maps, skipped {skipped} existing maps "
#         f"from {source_dir} to {output_dir}"
#     )


# if __name__ == "__main__":
#     main()